# EDA and Results Figures

Exploration only -- all business logic lives in `src/emot/`. This notebook:
1. Explores the MOT17 subset (sequence composition, sample annotated frames).
2. Loads `results/*.json` and renders the comparison figures referenced in `docs/results.md`.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from emot.data.mot17 import load_ground_truth, load_seqinfo

DATA_ROOT = Path("../data/mot17")
RESULTS_DIR = Path("../results")
CACHE_DIR = Path("../cache/detections")
FIGURES_DIR = Path("../docs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sequences = sorted(d.name for d in DATA_ROOT.glob("MOT17-*") if d.is_dir())
sequences

## 1. Sequence composition

In [ ]:
rows = []
for seq in sequences:
    info = load_seqinfo(DATA_ROOT / seq)
    gt = load_ground_truth(DATA_ROOT / seq)
    rows.append({
        "sequence": seq,
        "n_frames": info.seq_length,
        "frame_rate": info.frame_rate,
        "resolution": f"{info.im_width}x{info.im_height}",
        "n_unique_pedestrians": gt["id"].nunique(),
        "n_gt_boxes": len(gt),
        "mean_boxes_per_frame": len(gt) / info.seq_length,
    })
df = pd.DataFrame(rows)
df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df.set_index("sequence")["mean_boxes_per_frame"].plot.bar(ax=ax, color="#0097e6")
ax.set_ylabel("mean ground-truth boxes / frame")
ax.set_title("MOT17 subset: pedestrian density by sequence")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sequence_density.png", dpi=150)
plt.show()

## 2. Sample annotated frames per sequence

In [ ]:
fig, axes = plt.subplots(1, len(sequences), figsize=(5 * len(sequences), 5))
for ax, seq in zip(axes, sequences, strict=True):
    info = load_seqinfo(DATA_ROOT / seq)
    gt = load_ground_truth(DATA_ROOT / seq)
    mid_frame = info.seq_length // 2
    img = cv2.cvtColor(
        cv2.imread(str(DATA_ROOT / seq / "img1" / f"{mid_frame:06d}.jpg")), cv2.COLOR_BGR2RGB
    )
    ax.imshow(img)
    frame_gt = gt[gt["frame"] == mid_frame]
    for _, row in frame_gt.iterrows():
        ax.add_patch(
            plt.Rectangle(
                (row.bb_left, row.bb_top), row.bb_width, row.bb_height,
                fill=False, edgecolor="#00ff9f", linewidth=1.5,
            )
        )
    ax.set_title(f"{seq} (frame {mid_frame}, {len(frame_gt)} GT boxes)")
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sample_annotated_frames.png", dpi=150)
plt.show()

## 3. Tracking benchmark results (from `results/*.json`)

In [ ]:
records = []
for seq_dir in sorted(RESULTS_DIR.glob("MOT17-*")):
    for backbone_dir in sorted(seq_dir.glob("*")):
        for f in sorted(backbone_dir.glob("*.json")):
            records.append(json.loads(f.read_text()))
results_df = pd.DataFrame(records)
results_df

In [ ]:
pivot = results_df.pivot_table(index=["sequence", "backbone"], columns="tracker", values="mota")
fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot.bar(ax=ax)
ax.set_ylabel("MOTA")
ax.set_title("MOTA by (sequence, backbone), ByteTrack vs DeepSORT")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mota_by_sequence_backbone.png", dpi=150)
plt.show()

In [ ]:
id_switches = results_df.groupby("tracker")["num_switches"].agg(["mean", "sum"])
fig, ax = plt.subplots(figsize=(6, 4))
id_switches["mean"].plot.bar(ax=ax, color=["#0097e6", "#e84118"])
ax.set_ylabel("mean ID switches per run")
ax.set_title("Identity-switch comparison: ByteTrack vs DeepSORT")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "id_switches_by_tracker.png", dpi=150)
plt.show()
id_switches

## 4. Export benchmark (ONNX / INT8)

In [ ]:
export_path = RESULTS_DIR / "export_benchmark.json"
if export_path.exists():
    export_df = pd.DataFrame(json.loads(export_path.read_text()))
    pivot_export = export_df.pivot(index="backbone", columns="variant", values="mean_latency_ms")
    fig, ax = plt.subplots(figsize=(8, 5))
    pivot_export.plot.bar(ax=ax)
    ax.set_ylabel("mean latency (ms/frame, CPU)")
    ax.set_title("Edge export latency: PyTorch FP32 vs ONNX FP32 vs ONNX INT8")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "export_latency.png", dpi=150)
    plt.show()
    display(export_df)
else:
    print("No export benchmark yet -- run src/emot/training/run_export_benchmark.py")